# Phase 1 control: single condition

Same recipe as the mixed run in every respect except one. The changed
half is deterministic `silent_break` only, which is what the August
anchors were. `N_WORLDS`, `PRES_REPEAT`, `EPOCHS`, `LR`, `GRAD_ACCUM`,
the seeds and the training surface are all unchanged, so the condition
mix is the single variable.

The question it answers: the mixed adapter scored 5/32 on graded
localization where August scored 31/32. Was that the condition mix, or
something in the rewrite of `anchors.py` into `anchors_v22.py`?

* **Near 31/32** -> August reproduces under the new pipeline. The
  collapse is the mix, and that is a result: broad coverage costs 26
  points, which is evidence the 31/32 was one rule rather than a world
  model.
* **Near 5/32** -> the rewrite differs from the old generator in some
  other way, and that needs chasing before more GPU.

The mixed run, for comparison when the sanity table appears at the
bottom: sensitivity 4/4, specificity 2/4, localization 0/4,
preservation 2/8 all-unchanged, 28/28 bare JSON.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
%pip install -q -U transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.3 MB ? eta -:--:--^C
   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.2/12.3 MB 13.1 MB/s eta 0:00:01
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


## Preflight

Discovers the datasets, checks the GPU, and picks a compute dtype. **T4
is Turing and has no bfloat16**, so anything hard-coded to `bf16` fails
there. This picks `float16` on a T4 and `bfloat16` on an A100 or L4.

In [3]:
import os, sys, glob, json, collections
import torch

def find_dir(marker, root="/kaggle/input"):
    """Directory containing `marker`, at any depth under root."""
    hits = sorted(glob.glob(os.path.join(root, "**", marker), recursive=True),
                  key=lambda p: (p.count(os.sep), len(p)))
    if not hits:
        raise SystemExit(f"no {marker} under {root}")
    return os.path.dirname(hits[0])

REPO_PATH = find_dir("resource_mdp.py")
EVAL_PATH = find_dir("anchors_v22.py")
OUT_DIR   = "/kaggle/working"
print("repo:", REPO_PATH)
print("eval:", EVAL_PATH)

for f in ("run_pilot.py", "ecpm_parser.py", "explore_agent.py",
          "explore_metrics.py"):
    assert os.path.isfile(os.path.join(REPO_PATH, f)), (
        f"{f} missing from the repo dataset. run_pilot.py imports "
        "explore_agent and explore_metrics at module level since Christian's "
        "fork was merged, so a partial copy of the repo will not work.")

assert torch.cuda.is_available(), "no GPU: set Accelerator in the sidebar"
GPU = torch.cuda.get_device_name(0)
CAP = torch.cuda.get_device_capability(0)
# is_bf16_supported() counts emulation, so it returns True on a T4 (7.5),
# which has no native bfloat16. Go by compute capability instead: Ampere
# and later only.
USE_BF16 = CAP[0] >= 8
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"gpu: {GPU} | capability {CAP[0]}.{CAP[1]} | bf16: {USE_BF16} "
      f"| compute dtype: {DTYPE}")

import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)

repo: /kaggle/input/datasets/mazwyy/ecpm-repo/ecpm-efe
eval: /kaggle/input/datasets/mazwyy/ecpm-eval
gpu: Tesla T4 | capability 7.5 | bf16: False | compute dtype: torch.float16
transformers 5.0.0 | peft 0.19.1


In [4]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_OUT = f"{OUT_DIR}/anchor_adapter_singlecond_{MODEL_NAME.split('/')[-1]}" # was ADAPTER_OUT = f"{OUT_DIR}/anchor_adapter_{MODEL_NAME.split('/')[-1]}"

N_WORLDS     = 40
K            = 5
STOCH_SHARE  = 0.0       # was 0.5
PRES_REPEAT  = 1 # was 3
FIRST_SEED   = 1000
HOLDOUT_SEED = 2000      # sanity-check worlds, never trained on

EPOCHS     = 3
LR         = 1e-4
GRAD_ACCUM = 4 # was 8
MAX_LEN    = 2048
LORA_R, LORA_ALPHA = 16, 32
LORA_DROPOUT = 0.0       # dropout + gradient checkpointing raises
                         # CheckpointError, see the environment note
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
           "gate_proj", "up_proj", "down_proj"]

sys.path.insert(0, EVAL_PATH)
import anchors_v22, ecpm_eval as E
# single-condition control: 20 deterministic silent_break changed worlds,
# matching the August recipe. Everything else identical to the mixed run,
# so the only variable is the condition mix.
anchors_v22.CHANGED_CONDITIONS = ("silent_break",)
rp = anchors_v22.load_env(REPO_PATH)
E.attach(REPO_PATH)
import ecpm_parser as ep

ecpm_parser attached from /kaggle/input/datasets/mazwyy/ecpm-repo/ecpm-efe (looks like v2.2)


## Build the anchor set

Seeds start at 1000 and `build_anchor_set` asserts they clear the
official 0-79 range, so an anchor world cannot be a graded world.

In [5]:
worlds = anchors_v22.build_anchor_set(
    rp, n_worlds=N_WORLDS, k=K, stochastic_share=STOCH_SHARE,
    first_seed=FIRST_SEED, preservation_changed_repeat=PRES_REPEAT)
examples = anchors_v22.to_examples(worlds)

summary = anchors_v22.summarise(worlds)
for key, val in summary.items():
    print(f"  {key}: {val}")
json.dump(worlds, open(f"{OUT_DIR}/anchor_worlds.json", "w"))

assert summary["distinct_start_goal"] > 5, \
    "anchors collapsed onto too few start/goal pairs; that was the old bug"

# STOCH_SHARE controls this deliberately, so assert the shape that was
# asked for rather than assuming a mix
if STOCH_SHARE == 0.0:
    assert summary["stochastic_worlds"] == 0, "expected deterministic only"
else:
    assert summary["stochastic_worlds"] > 0, "no stochastic anchors"

want = set(anchors_v22.CHANGED_CONDITIONS) | {"no_change"}
assert set(summary["conditions"]) <= want, \
    f"unexpected conditions {set(summary['conditions']) - want}"
matching = summary["conditions"].get("silent_break", 0)
print(f"\nchanged worlds: {summary['changed_worlds']} | conditions "
      f"{summary['conditions']}")
print(f"deterministic silent_break worlds: "
      f"{matching if STOCH_SHARE == 0.0 else 'mixed'}  "
      f"(mixed run had 3, August had 20)")

  worlds: 40
  examples: 140
  changed_worlds: 20
  deterministic_worlds: 40
  stochastic_worlds: 0
  conditions: {'silent_break': 20, 'no_change': 20}
  examples_by_probe: {'detection': 40, 'localization': 20, 'preservation': 40, 'adaptation': 40}
  distinct_start_goal: 30
  preservation_pair_positive_rate: 0.125
  all_unchanged_would_score: 0.875
  seed_range: [1000, 1045]

changed worlds: 20 | conditions {'silent_break': 20, 'no_change': 20}
deterministic silent_break worlds: 20  (mixed run had 3, August had 20)


## Verify the gold before training on it

Every gold answer must score correct through `ecpm_parser`. If this
fails, phase 1 would teach the model to produce answers the shared scorer
marks wrong. Should print 180 checked, no failures.

In [6]:
bad, checked = collections.Counter(), 0
for w in worlds:
    sc = anchors_v22._scenario(w["seed"], w["condition"], K)
    rec = rp.build_record(sc, w["deterministic"])
    for it in w["items"]:
        p = it["probe"]
        if p not in ("detection", "localization", "preservation", "adaptation"):
            continue
        s = ep.run_probe(rec, p, it["gold"],
                         queried_pairs=(w["queried_pairs"]
                                        if p == "preservation" else None))["scored"]
        checked += 1
        if not (s.get("correct") is True or s.get("accuracy") == 1.0
                or s.get("is_optimal") is True):
            bad[p] += 1
print(f"gold answers scored: {checked}, failures: {dict(bad) or 'none'}")
assert not bad, "anchor gold does not score correct; do not train on this"

gold answers scored: 140, failures: none


## Tokenize, loss masked to the answer

The August adapter trained on raw tokenized text while inference went
through `apply_chat_template`, so it learned a distribution the model is
never in at test time. `SYSTEM` comes from `ecpm_eval`, the same constant
the eval sends.

`MAX_LEN` is asserted against the longest example. A truncated gold
answer leaves nothing behind the mask and contributes no gradient,
silently. Expect a max around 1100 tokens.

In [7]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def as_ids(x):
    """Token ids out of whatever apply_chat_template returns.

    In transformers 5.x it can hand back a BatchEncoding, which subclasses
    UserDict rather than dict, so an isinstance(x, dict) test is False and
    list(x) silently yields the KEYS. Check for input_ids directly instead,
    then assert the result really is ints.
    """
    if hasattr(x, "input_ids"):
        x = x.input_ids
    elif hasattr(x, "keys") and "input_ids" in x.keys():
        x = x["input_ids"]
    x = list(x)
    if x and isinstance(x[0], (list, tuple)):
        x = list(x[0])
    if not all(isinstance(t, int) for t in x):
        raise TypeError(f"expected token ids, got {type(x[0]).__name__}: {x[:4]}")
    return x

def encode(ex):
    prefix = as_ids(tok.apply_chat_template(
        [{"role": "system", "content": E.SYSTEM},
         {"role": "user", "content": ex["prompt"]}],
        add_generation_prompt=True, tokenize=True))
    answer = as_ids(tok(ex["gold"] + tok.eos_token,
                        add_special_tokens=False)["input_ids"])
    return {"input_ids": prefix + answer,
            "labels": [-100] * len(prefix) + answer,
            "n_prefix": len(prefix), "n_answer": len(answer)}

enc = [encode(e) for e in examples]
lengths = sorted(len(x["input_ids"]) for x in enc)
prefixes = sorted(x["n_prefix"] for x in enc)
print(f"{len(enc)} examples | tokens min {lengths[0]} "
      f"median {lengths[len(lengths)//2]} max {lengths[-1]}")
print(f"prefix tokens min {prefixes[0]} max {prefixes[-1]}")
print(f"answer tokens min {min(x['n_answer'] for x in enc)} "
      f"max {max(x['n_answer'] for x in enc)}")
print("first 8 ids:", enc[0]["input_ids"][:8])

assert lengths[-1] <= MAX_LEN, (
    f"longest example is {lengths[-1]} tokens but MAX_LEN is {MAX_LEN}")
assert prefixes[0] > 300, (
    f"shortest prompt is only {prefixes[0]} tokens; the evidence block alone "
    "is ~700, so the chat template did not tokenize properly")
assert all(len(x["input_ids"]) == len(x["labels"]) for x in enc)
assert all(any(l != -100 for l in x["labels"]) for x in enc)
assert all(isinstance(t, int) for x in enc for t in x["input_ids"])

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

140 examples | tokens min 1312 median 1484 max 1748
prefix tokens min 1306 max 1671
answer tokens min 6 max 77
first 8 ids: [151644, 8948, 198, 2610, 525, 41018, 16230, 18422]


In [8]:
from torch.utils.data import Dataset

class Anchors(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return {"input_ids": r["input_ids"], "labels": r["labels"]}

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        gap = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * gap)
        out["labels"].append(b["labels"] + [-100] * gap)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * gap)
    return {k: torch.tensor(v) for k, v in out.items()}

train_ds = Anchors(enc)
print(len(train_ds), "training examples")

140 training examples


## Model

Pinned to device 0 rather than `device_map="auto"`. A 1.5B model in 4-bit
fits one T4 comfortably, and sharding it across two confuses `Trainer`.

In [9]:
from transformers import (AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map={"": 0})

# Checkpointing ON. The environment note disabled it because LoRA dropout
# makes the recompute pass save a different tensor count and raise
# CheckpointError. LORA_DROPOUT is 0.0 here, so there is nothing stochastic
# to mismatch, and without it a 1748-token example OOMs a 14.5 GB T4.
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM", target_modules=TARGETS))
model.print_trainable_parameters()
assert LORA_DROPOUT == 0.0, "checkpointing needs dropout 0.0"

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

## Step math, checked before training

The August run chunked one evidence string into 3 examples and ran
`num_train_epochs=20` with `gradient_accumulation_steps=4`. `3 // 4 = 0`,
clamped to 1, so 20 epochs produced 20 optimizer steps. Expect about 66
here.

In [ ]:
steps_per_epoch = max(1, len(train_ds) // GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
print(f"{len(train_ds)} examples / grad_accum {GRAD_ACCUM} "
      f"= {steps_per_epoch} steps per epoch x {EPOCHS} epochs "
      f"= {total_steps} optimizer steps")
assert len(train_ds) >= GRAD_ACCUM, (
    f"{len(train_ds)} examples with grad_accum {GRAD_ACCUM} rounds to zero "
    "steps per epoch; lower GRAD_ACCUM or add worlds")
assert total_steps >= 20, f"only {total_steps} optimizer steps"

args = TrainingArguments(
    output_dir=f"{OUT_DIR}/phase1",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_steps=max(1, total_steps // 20),   # warmup_ratio is gone in 5.x
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="no",
    report_to=[],
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=collate)
result = trainer.train()
print(result.metrics)

## Save, with provenance

`adapter_config.json` is what the eval notebook asserts against, so the
base model name and dropout recorded here are what stop a 3B adapter
being loaded on a 1.5B base later. The last adapter could not be
identified from its own files, which is why the provenance file
exists.

In [ ]:
model.save_pretrained(ADAPTER_OUT)
tok.save_pretrained(ADAPTER_OUT)

prov = {"model": MODEL_NAME, "phase": 1, "gpu": GPU, "dtype": str(DTYPE),
        "transformers": transformers.__version__, "peft": peft.__version__,
        "anchor_worlds": len(worlds), "anchor_examples": len(examples),
        "first_seed": FIRST_SEED, "k": K, "stochastic_share": STOCH_SHARE,
        "preservation_repeat": PRES_REPEAT,
        "changed_conditions": list(anchors_v22.CHANGED_CONDITIONS),
        "run": "single_condition_control",
        "epochs": EPOCHS, "lr": LR, "grad_accum": GRAD_ACCUM,
        "optimizer_steps": total_steps,
        "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
                 "targets": TARGETS},
        "final_loss": result.metrics.get("train_loss"),
        "anchor_summary": summary}
json.dump(prov, open(f"{ADAPTER_OUT}/phase1_provenance.json", "w"), indent=1)

cfg = json.load(open(f"{ADAPTER_OUT}/adapter_config.json"))
assert cfg["base_model_name_or_path"] == MODEL_NAME, cfg["base_model_name_or_path"]
assert cfg["lora_dropout"] == 0.0, cfg["lora_dropout"]
print("saved to", ADAPTER_OUT)
print(json.dumps(prov, indent=1))

## Sanity check on held-out anchor worlds

Seeds 2000+, never trained on. This is a format-and-transfer check, not a
result. What it answers:

* does the adapter emit parseable JSON on all four probes
* does it still answer "nothing changed" on every preservation probe, the
  failure `PRES_REPEAT` was meant to break
* does it hold up on stochastic worlds, which the old anchors never
  covered

Takes a few minutes. The graded evaluation is `armc_eval_v22.ipynb` on
the official payloads.

In [ ]:
import pandas as pd

hold = anchors_v22.build_anchor_set(
    rp, n_worlds=8, k=K, stochastic_share=STOCH_SHARE,
    first_seed=HOLDOUT_SEED, preservation_changed_repeat=1)
assert not ({w["seed"] for w in hold} & {w["seed"] for w in worlds})

model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

def ask(messages, max_new_tokens):
    e = tok.apply_chat_template(messages, add_generation_prompt=True,
                                return_tensors="pt", return_dict=True)
    e = {k: v.to(model.device) for k, v in e.items()}
    with torch.no_grad():
        o = model.generate(**e, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(o[0, e["input_ids"].shape[1]:], skip_special_tokens=True)

rows = []
for w in hold:
    sc = anchors_v22._scenario(w["seed"], w["condition"], K)
    rec = rp.build_record(sc, w["deterministic"])
    seen = set()
    for it in w["items"]:
        p = it["probe"]
        if p in seen or p not in ("detection", "localization",
                                  "preservation", "adaptation"):
            continue
        seen.add(p)
        raw = ask([{"role": "system", "content": E.SYSTEM},
                   {"role": "user", "content": it["prompt"]}],
                  E.MAX_NEW_TOKENS.get(p, 400))
        res = ep.run_probe(rec, p, raw,
                           queried_pairs=(w["queried_pairs"]
                                          if p == "preservation" else None))
        rows.append({"arm": "arm_c", "seed": w["seed"],
                     "condition": w["condition"],
                     "deterministic": w["deterministic"], "mode": "single",
                     "probe": p, "turn": 2, "prompt": it["prompt"], "raw": raw,
                     "bare_json": E.is_bare_json(raw), "target": w["target"],
                     "parsed": res["parsed"], "scored": res["scored"]})
    print(f"  seed {w['seed']} {w['condition']} done")

with open(f"{OUT_DIR}/phase1_singlecond_sanity_rows.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")

tab = E.table(rows, arms=["arm_c"])
display(pd.DataFrame(tab)[["arm", "sens", "spec", "localize", "pres_acc",
                           "pres_const", "target_recall", "pres_parsed",
                           "route_valid", "route_optimal", "bare_json"]])
print("route status:", tab[0]["route_status"])

In [ ]:
pres = [r for r in rows if r["probe"] == "preservation"]
const = sum(1 for r in pres
            if r["parsed"].get("status") == "ok"
            and not any(p["changed"] for p in r["parsed"]["pairs"]))
print(f"all-unchanged preservation replies: {const}/{len(pres)}")
if const == len(pres):
    print("\nSTILL CONSTANT. The adapter answers 'nothing changed' on every "
          "preservation probe, same as the old one. Raise PRES_REPEAT and "
          "rerun, or treat judgement preservation as unreportable for this "
          "arm and use belief_self_consistency from a two-turn run instead.")
else:
    print("\nNot constant. Judgement preservation is worth reporting.")

print("\nSend these three things back:")
print("  1. final_loss:", result.metrics.get("train_loss"))
print("  2. the sanity table above")
print(f"  3. all-unchanged preservation replies: {const}/{len(pres)}")

In [ ]:
#### import os, glob, zipfile, shutil

ZIP_PATH = f"{OUT_DIR}/phase1_singlecond_all.zip"
SKIP = ("phase1/", "phase1_singlecond_all.zip", ".ipynb_checkpoints")

# the Trainer output dir is empty with save_strategy="no", and any
# checkpoint-* folders in it would double the zip for nothing
shutil.rmtree(f"{OUT_DIR}/phase1", ignore_errors=True)

n = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(glob.glob(f"{OUT_DIR}/**/*", recursive=True)):
        if not os.path.isfile(path):
            continue
        rel = os.path.relpath(path, OUT_DIR)
        if any(rel.startswith(s) or s in rel for s in SKIP):
            continue
        z.write(path, rel)
        n += 1

print(f"{n} files -> {ZIP_PATH}  ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)\n")
with zipfile.ZipFile(ZIP_PATH) as z:
    for i in z.namelist():
        print("  ", i)